## Система AI-учителя

In [1]:
!pip install openai==0.27

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.1/70.1 kB 2.3 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.54.4
    Uninstalling openai-1.54.4:
      Successfully uninstalled openai-1.54.4


In [2]:
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.1 MB/s eta 0:00:00


In [3]:
from google.colab import userdata

In [4]:
from IPython.display import display, Markdown, HTML
import ipywidgets as widgets
import openai
from datetime import datetime


class ColabTeacher:
    def __init__(self, api_key=None):
        self.api_key = api_key
        if api_key:
            openai.api_key = api_key
        self.create_widgets()
        self.conversation_history = []

    def create_widgets(self):
        # Поле ввода вопроса
        self.question_input = widgets.Textarea(
            placeholder='Введите ваш вопрос...',
            layout={'width': '600px', 'height': '50px'}
        )

        # Добавляем обработчик клавиши Enter
        self.question_input.observe(self.handle_keypress, names='value')

        # Кнопка отправки
        self.submit_button = widgets.Button(
            description='Задать вопрос',
            button_style='primary',
            layout={'width': 'auto', 'margin': '10px 0'}
        )
        self.submit_button.on_click(self.handle_question)

        # Область вывода
        self.output_area = widgets.Output()

    def handle_keypress(self, change):
        if change.new.endswith('\n'):
            # Очищаем поле ввода сразу
            question = change.new.rstrip()  # Убираем последний перенос строки
            self.question_input.value = ''

            # Если вопрос не пустой (кроме пробелов и переносов строк), обрабатываем его
            if question.strip():
                self.process_question(question)

    def display_interface(self):
        display(HTML("""
            <div style="margin: 20px 0;">
                <h2>🤖 AI-учитель</h2>
                <p style="color: #666;">
                    Enter — отправить вопрос<br>
                    Shift+Enter — перенос строки
                </p>
            </div>
        """))

        display(self.question_input)
        display(self.submit_button)
        display(self.output_area)

    def handle_question(self, _):
        question = self.question_input.value
        if question.strip():
            self.question_input.value = ''
            self.process_question(question)

    def process_question(self, question):
        try:
            current_time = datetime.now().strftime("%d.%m.%Y %H:%M")

            self.conversation_history.append({
                "role": "user",
                "content": question,
                "time": current_time
            })


            # TODO: написать промпт
            prompt = "Ты - AI-учитель, помогающий изучать основы искусственного интеллекта."

            response = openai.ChatCompletion.create(
                model="gpt-3.5-turbo",
                messages=[
                    {"role": "system", "content": prompt},
                    *[{"role": msg["role"], "content": msg["content"]} for msg in self.conversation_history]
                ]
            )

            answer = response.choices[0].message.content
            self.conversation_history.append({
                "role": "assistant",
                "content": answer,
                "time": current_time
            })

            self.output_area.clear_output()

            # Добавляем все сообщения в обратном порядке
            with self.output_area:
                for i in range(len(self.conversation_history)-1, -1, -2):
                    if i >= 1:  # Проверяем, что есть пара вопрос-ответ
                        q = self.conversation_history[i-1]["content"]
                        a = self.conversation_history[i]["content"]
                        time = self.conversation_history[i]["time"]

                        # Определяем стили для текущего блока
                        if i == len(self.conversation_history) - 1:
                            border_color = "#007bff"  # Синий для последнего ответа
                            time_color = "#007bff"
                        else:
                            border_color = "#6c757d"  # Серый для старых ответов
                            time_color = "#6c757d"

                        display(HTML(f"""
                            <div style="margin: 20px 0; border-left: 3px solid {border_color};">
                                <!-- Вопрос -->
                                <div style="padding: 10px; background-color: #f8f9fa; margin-bottom: 1px;">
                                    {q.replace(chr(10), '<br>')}
                                </div>

                                <!-- Ответ -->
                                <div style="padding: 10px; background-color: #ffffff;">
                                    {a.replace(chr(10), '<br>')}
                                    <div style="margin-top: 10px; font-size: 0.8em; color: {time_color};">
                                        {time}
                                    </div>
                                </div>
                            </div>
                        """))

        except Exception as e:
            with self.output_area:
                display(HTML(f"""
                    <div style="color: #dc3545; margin: 10px 0;">
                        ❌ Произошла ошибка: {str(e)}
                    </div>
                """))

In [ ]:
# TODO: Составляем программу обучения

In [ ]:
# TODO: сохранять состояние между уроками = перенос данных в новый ноутбук
# - в ноутбуке
# - в файле
# - онлайн (Google Drive)

# Ai Teacher Data Handling
import json
from google.colab import drive
import os

# Подключение Google Drive
class DataManager:
    def __init__(self):
        self.drive_mounted = False
        self.data_file = "ai_teacher_data.json"
        self.data = {
            "course": {},
            "student": {
                "name": "",
                "gender": "",
                "preferred_address": ""
            },
            "progress": {
                "completed_topics": [],
                "completed_tasks": [],
                "current_state": ""
            }
        }

    def mount_drive(self):
        if not self.drive_mounted:
            drive.mount('/content/drive')
            self.drive_mounted = True

    def set_file_path(self, folder_path="/content/drive/My Drive/AI_Teacher/"):
        self.folder_path = folder_path
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)
        self.full_path = os.path.join(folder_path, self.data_file)

    def save_data(self):
        try:
            with open(self.full_path, "w") as f:
                json.dump(self.data, f, indent=4)
            print(f"Данные успешно сохранены в {self.full_path}")
        except Exception as e:
            print(f"Ошибка при сохранении данных: {e}")

    def load_data(self):
        try:
            if os.path.exists(self.full_path):
                with open(self.full_path, "r") as f:
                    self.data = json.load(f)
                print("Данные успешно загружены.")
            else:
                print("Файл данных не найден, создается новый файл.")
                self.save_data()
        except Exception as e:
            print(f"Ошибка при загрузке данных: {e}")

    def update_student_info(self, name, gender, preferred_address):
        self.data["student"] = {
            "name": name,
            "gender": gender,
            "preferred_address": preferred_address
        }
        self.save_data()

    def update_progress(self, completed_topic=None, completed_task=None, current_state=None):
        if completed_topic:
            self.data["progress"]["completed_topics"].append(completed_topic)
        if completed_task:
            self.data["progress"]["completed_tasks"].append(completed_task)
        if current_state:
            self.data["progress"]["current_state"] = current_state
        self.save_data()

# Пример использования
manager = DataManager()
manager.mount_drive()
manager.set_file_path()
manager.load_data()

# Обновление информации об обучающемся
manager.update_student_info(name="Иван Иванов", gender="мужской", preferred_address="вы")

# Обновление прогресса
manager.update_progress(completed_topic="Введение в машинное обучение", current_state="Изучает линейную регрессию")
import json
from google.colab import drive
import os

# Подключение Google Drive
class DataManager:
    def __init__(self):
        self.drive_mounted = False
        self.data_file = "ai_teacher_data.json"
        self.data = {
            "course": {},
            "student": {
                "name": "",
                "gender": "",
                "preferred_address": ""
            },
            "progress": {
                "completed_topics": [],
                "completed_tasks": [],
                "current_state": ""
            }
        }

    def mount_drive(self):
        if not self.drive_mounted:
            drive.mount('/content/drive')
            self.drive_mounted = True

    def set_file_path(self, folder_path="/content/drive/My Drive/AI_Teacher/"):
        self.folder_path = folder_path
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)
        self.full_path = os.path.join(folder_path, self.data_file)

    def save_data(self):
        try:
            with open(self.full_path, "w") as f:
                json.dump(self.data, f, indent=4)
            print(f"Данные успешно сохранены в {self.full_path}")
        except Exception as e:
            print(f"Ошибка при сохранении данных: {e}")

    def load_data(self):
        try:
            if os.path.exists(self.full_path):
                with open(self.full_path, "r") as f:
                    self.data = json.load(f)
                print("Данные успешно загружены.")
            else:
                print("Файл данных не найден, создается новый файл.")
                self.save_data()
        except Exception as e:
            print(f"Ошибка при загрузке данных: {e}")

    def update_student_info(self, name, gender, preferred_address):
        self.data["student"] = {
            "name": name,
            "gender": gender,
            "preferred_address": preferred_address
        }
        self.save_data()

    def update_progress(self, completed_topic=None, completed_task=None, current_state=None):
        if completed_topic:
            self.data["progress"]["completed_topics"].append(completed_topic)
        if completed_task:
            self.data["progress"]["completed_tasks"].append(completed_task)
        if current_state:
            self.data["progress"]["current_state"] = current_state
        self.save_data()

# Пример использования
manager = DataManager()
manager.mount_drive()
manager.set_file_path()
manager.load_data()

# Обновление информации об обучающемся
manager.update_student_info(name="Иван Иванов", gender="мужской", preferred_address="вы")

# Обновление прогресса
manager.update_progress(completed_topic="Введение в машинное обучение", current_state="Изучает линейную регрессию")


In [ ]:
# TODO: способность задавать проверочные вопросы
# - создавать задания для ученика
# - давать возможность их выполнить и предъявить решение и ответ
# - проверить и написать отзыв для ученика

In [ ]:
# TODO: способность приводить примеры кода, которые можно запускать
# TODO: графики (matplotlib, Plotly...)

In [ ]:
# TODO: дать возможность скачать результаты

In [ ]:
# TODO: в будущем
# - Улучить интерфейс - "думаю..." и т.д.
# - Выводить ответы, отформатированные красиво, например, образцы кода
# - Что сохраняем между уроками:
#   - Программа обучения
#   - Прогресс
#       - Выполненыые задания
#       - Частые ошибки
#       - ...
#   - Сведения об ученике: уровень, стиль общения...

## Разговор с AI-учителем

In [5]:
teacher = ColabTeacher(userdata.get('OPENAI_API_KEY'))
teacher.display_interface()

Textarea(value='', layout=Layout(height='50px', width='600px'), placeholder='Введите ваш вопрос...')

Button(button_style='primary', description='Задать вопрос', layout=Layout(margin='10px 0', width='auto'), styl…

Output()

In [ ]:
my_list = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]
sorted_list = sorted(my_list)
print(sorted_list)

[1, 1, 2, 3, 3, 4, 5, 5, 5, 6, 9]


In [ ]:
text = """
Использование метода `sorted()` для создания нового отсортированного списка:

```python
my_list = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]
sorted_list = sorted(my_list)
print(sorted_list)
```

Использование метода `sort()` для сортировки списка на месте (не создает новый отсортированный список):

```python
my_list = [3, 1, 4, 1, 5, 9, 2]
my_list.sort()
print(my_list)
```
"""

display(Markdown(text))


Использование метода `sorted()` для создания нового отсортированного списка:

```python
my_list = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]
sorted_list = sorted(my_list)
print(sorted_list)
```

Использование метода `sort()` для сортировки списка на месте (не создает новый отсортированный список):

```python
my_list = [3, 1, 4, 1, 5, 9, 2]
my_list.sort()
print(my_list)
```
